# Anexo II — Obtención y preparación de los datos

Códigos A2.1–A2.61 extraídos de la versión final del TFM.

## A2.1. Conexión con la API de Idealista


In [ ]:
import requests
import base64
# Definición de las credenciales de acceso
# (Ocultas por motivos de seguridad)
api_key = "********************"
api_secret = "********************"
# Construcción de la cabecera de autenticación
credentials = f"{api_key}:{api_secret}"
encoded_credentials = base64.b64encode(
credentials.encode()
).decode()
headers = {
"Authorization": f"Basic {encoded_credentials}",
"Content-Type": "application/x-www-form-urlencoded"
}
# Solicitud del Bearer Token
data = {
"grant_type": "client_credentials"
}
response = requests.post(
"https://api.idealista.com/oauth/token",
headers=headers,
data=data
)
# Extracción del token de acceso
token = response.json()["access_token"]


## A2.2. Configuración de los parámetros de búsqueda


In [ ]:
# Endpoint de búsqueda
url = "https://api.idealista.com/3.5/es/search"
# Parámetros de búsqueda
payload = {
"center": "40.474,-3.642",
"distance": 500,
"propertyType": "homes",
"operation": "sale",
"maxItems": 50
}


## A2.3. Paginación de los resultados


In [ ]:
# Descarga y consolidación de los resultados
# mediante paginación
# Primera petición para obtener los resultados

# y el número total de páginas
response = requests.post(
url,
headers=headers,
data=payload
)
data = response.json()
# Lista para almacenar todos los anuncios
todos_los_anuncios = []
# Incorporación de los resultados de la primera página
todos_los_anuncios.extend(
data["elementList"]
)
# Número total de páginas disponibles
total_paginas = data["totalPages"]
# Recorrido del resto de páginas
for pagina in range(2, total_paginas + 1):
payload["numPage"] = pagina
response = requests.post(
url,
headers=headers,
data=payload
)
respuesta = response.json()
todos_los_anuncios.extend(
respuesta["elementList"]
)


## A2.4. Transformación de los resultados a DataFrame


In [ ]:
import pandas as pd
# Conversión de los anuncios a estructura tabular
df = pd.DataFrame(todos_los_anuncios)
# Visualización inicial
df.head()


## A2.5. Eliminación de duplicados, selección del área y exportación del dataset


In [ ]:
df = df.drop_duplicates(
subset="propertyCode"
)
# Selección de las viviendas pertenecientes
# al distrito de Hortaleza
df = df[
df["district"].str.lower() == "hortaleza"
]

# Exportación del dataset
df.to_csv(
"dataset_base_hortaleza.csv",
index=False,
encoding="utf-8-sig"
)
print(
f"Dataset generado: {len(df)} viviendas"
)


## A2.6. Carga y comprobación del dataset inmobiliario


In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv("dataset_base_hortaleza.csv")
print("Dimensiones del dataset:", df.shape)
print("\nNúmero de viviendas:", len(df))
print("\nColumnas disponibles:")
print(df.columns.tolist())
print("\nValores nulos en las coordenadas:")
print(df[["latitude", "longitude"]].isnull().sum())
print("\nAnuncios duplicados por propertyCode:")
print(df["propertyCode"].duplicated().sum())


## A2.7. Creación del GeoDataFrame a partir de las coordenadas de las viviendas


In [ ]:
gdf = gpd.GeoDataFrame(
df.copy(),
geometry=gpd.points_from_xy(
df["longitude"],
df["latitude"]
),
crs="EPSG:4326"
)
print("Tipo de objeto:", type(gdf))
print("Sistema de referencia:", gdf.crs)
print("Número de viviendas:", len(gdf))
print("\nGeometría de las primeras viviendas:")
print(gdf.geometry.head())


## A2.8. Reproyección de las viviendas al sistema EPSG:25830


In [ ]:
gdf_utm = gdf.to_crs("EPSG:25830")
print("Sistema de referencia original:", gdf.crs)
print("Sistema de referencia reproyectado:", gdf_utm.crs)
print("Número de viviendas:", len(gdf_utm))
print("\nGeometría de las primeras viviendas:")

print(gdf_utm.geometry.head())


## A2.9. Carga y comprobación del límite administrativo del distrito de Hortaleza


In [ ]:
hortaleza = gpd.read_file("Dist_Hortaleza.shp")
print("Shapefile cargado correctamente.")
print("\nNúmero de geometrías:", len(hortaleza))
print("\nCRS:", hortaleza.crs)
print("\nTipo de geometría:")
print(hortaleza.geometry.geom_type.value_counts())
print("\nColumnas disponibles:")
print(hortaleza.columns.tolist())
print("\nPrimeros registros:")
print(hortaleza.head())
# Reproyección del shapefile de Hortaleza
hortaleza_utm = hortaleza.to_crs("EPSG:25830")
print("CRS original:", hortaleza.crs)
print("CRS reproyectado:", hortaleza_utm.crs)


## A2.10. Creación del área de consulta para la obtención de información de OpenStreetMap


In [ ]:
area_estudio = gdf_utm.geometry.union_all().convex_hull
area_osm = area_estudio.buffer(2000)
gdf_area_osm = gpd.GeoDataFrame(
{"nombre": ["Área de consulta OSM"]},
geometry=[area_osm],
crs=gdf_utm.crs
)
print("Área de consulta creada correctamente.")
print("Sistema de referencia:", gdf_area_osm.crs)
print("Número de geometrías:", len(gdf_area_osm))
print("\nTipo de geometría:")
print(gdf_area_osm.geometry.geom_type.iloc[0])
print("\nÁrea de consulta (km²):")
print(gdf_area_osm.geometry.area.iloc[0] / 1_000_000)
# Conversión del área de consulta a WGS84 para OSMnx
poligono_osm = gdf_area_osm.to_crs(
"EPSG:4326"
).geometry.iloc[0]


## A2.11. Consulta y comprobación de elementos geográficos obtenidos de OpenStreetMap


In [ ]:
import osmnx as ox
metro = ox.features_from_polygon(
poligono_osm,

tags={"railway": "station"}
)
print("Elementos ferroviarios encontrados:", len(metro))
print("\nCRS de los datos OSM:")
print(metro.crs)
print("\nValores de railway:")
print(metro["railway"].value_counts(dropna=False))
print("\nValores de station:")
print(metro["station"].value_counts(dropna=False))


## A2.12. Selección y revisión de las estaciones de Metro obtenidas de OpenStreetMap


In [ ]:

metro = metro[
metro["station"] == "subway"
].copy()
print("Estaciones de Metro:", len(metro))
print("\nNombres de las estaciones de Metro:")
print(
metro["name"].sort_values().to_string(index=False)
)
print("\nEstaciones sin nombre:")
print(
metro[metro["name"].isna()][["name", "geometry"]]
)
print("\nNombres duplicados:")
print(
metro[
metro["name"].notna() &
metro["name"].duplicated(keep=False)
][["name", "geometry"]].sort_values("name")
)
print("\nTipos de geometría:")
print(
metro.geometry.geom_type.value_counts()
)


## A2.13. Limpieza, normalización y reproyección de las estaciones de Metro


In [ ]:
metro_limpio = metro.copy()
# 1. Eliminar elementos sin nombre
metro_limpio = metro_limpio[
metro_limpio["name"].notna()
].copy()
# 2. Convertir los polígonos en puntos representativos
metro_limpio["geometry"] = metro_limpio.geometry.apply(

lambda geom: geom if geom.geom_type == "Point"
else geom.representative_point()
)
# 3. Priorizar las geometrías Point
metro_limpio["tipo_geometria"] = metro_limpio.geometry.geom_type
metro_limpio = (
metro_limpio
.sort_values(
by=["name", "tipo_geometria"],
key=lambda x: x.map({
"Point": 0,
"Polygon": 1
}) if x.name == "tipo_geometria" else x
)
)
metro_limpio = metro_limpio.drop_duplicates(
subset="name",
keep="first"
).copy()
print("Número de estaciones después de la limpieza:",
len(metro_limpio))
print("\nTipos de geometría:")
print(
metro_limpio.geometry.geom_type.value_counts()
)
# Reproyección de las estaciones de Metro
metro_utm = metro_limpio.to_crs("EPSG:25830")
print("CRS original:", metro_limpio.crs)
print("CRS reproyectado:", metro_utm.crs)
print("\nNúmero de estaciones:", len(metro_utm))
print("\nTipos de geometría:")
print(
metro_utm.geometry.geom_type.value_counts()
)


## A2.14. Función para la obtención de variables de distancia a elementos de OpenStreetMap


In [ ]:

def calcular_distancia_osm(
gdf_viviendas,
poligono_osm,
tags,

nombre_variable
):
# 1. Consulta de elementos OSM
elementos = ox.features_from_polygon(
poligono_osm,
tags=tags
)
print(f"\n{nombre_variable}")
print("Elementos OSM encontrados:", len(elementos))
# 2. Eliminar elementos sin geometría
elementos = elementos[
elementos.geometry.notna()
].copy()
# 3. Convertir geometrías a puntos
elementos["geometry"] = elementos.geometry.apply(
lambda geom: geom if geom.geom_type == "Point"
else geom.representative_point()
)
# 4. Eliminar geometrías vacías
elementos = elementos[
~elementos.geometry.is_empty
].copy()
# 5. Reproyección a EPSG:25830
elementos_utm = elementos.to_crs("EPSG:25830")
# 6. Calcular distancia mínima desde cada vivienda
distancias = gdf_viviendas.geometry.apply(
lambda vivienda:
elementos_utm.geometry.distance(vivienda).min()
)
# 7. Incorporar variable al GeoDataFrame
gdf_viviendas[nombre_variable] = distancias
print(
"Valores nulos:",
gdf_viviendas[nombre_variable].isna().sum()
)
print(
"Distancia mínima (m):",
gdf_viviendas[nombre_variable].min()
)
print(
"Distancia máxima (m):",
gdf_viviendas[nombre_variable].max()
)
print(
"Distancia media (m):",

gdf_viviendas[nombre_variable].mean()
)
return gdf_viviendas


## A2.15. Cálculo de las distancias mínimas a los elementos urbanos considerados


In [ ]:
distancias_metro = gdf_utm.geometry.apply(
lambda vivienda:
metro_utm.geometry.distance(vivienda).min()
)
gdf_utm["dist_metro"] = distancias_metro
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"highway": "bus_stop"},
nombre_variable="dist_bus"
)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"amenity": "school"},
nombre_variable="dist_school"
)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"amenity": "hospital"},
nombre_variable="dist_hospital"
)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"amenity": "clinic"},
nombre_variable="dist_health_center"
)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"shop": "supermarket"},
nombre_variable="dist_supermarket"
)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"leisure": "park"},
nombre_variable="dist_park"

)
gdf_utm = calcular_distancia_osm(
gdf_viviendas=gdf_utm,
poligono_osm=poligono_osm,
tags={"leisure": "sports_centre"},
nombre_variable="dist_sports"
)


## A2.16. Integración de las ocho variables geoespaciales de accesibilidad


In [ ]:
variables_osm = [
"dist_metro",
"dist_bus",
"dist_school",
"dist_hospital",
"dist_health_center",
"dist_supermarket",
"dist_park",
"dist_sports"
]
print("Variables geoespaciales incorporadas:")
print(variables_osm)
print("\nNúmero de viviendas:", len(gdf_utm))


## A2.17. Control de calidad de las variables geoespaciales de accesibilidad


In [ ]:
print("Número de viviendas:", len(gdf_utm))
print("\nValores nulos:")
print(
gdf_utm[variables_osm].isnull().sum()
)
print("\nValores iguales a cero:")
print(
(gdf_utm[variables_osm] == 0).sum()
)
print("\nResumen estadístico:")
print(
gdf_utm[variables_osm].describe().T[
["min", "mean", "50%", "max"]
]
)


## A2.18. Representación cartográfica de las viviendas analizadas, el límite del distrito de Hortaleza y las estaciones de metro utilizadas 


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 10))

# Límite del distrito
hortaleza_utm.boundary.plot(
ax=ax,
linewidth=1,
label="Límite de Hortaleza"
)
# Viviendas
gdf_utm.plot(
ax=ax,
markersize=8,
label="Viviendas"
)
# Estaciones de Metro
metro_utm.plot(
ax=ax,
markersize=35,
marker="^",
label="Estaciones de Metro"
)
ax.set_title(
"Viviendas de Hortaleza y estaciones de Metro"
)
# Leyenda
ax.legend(
loc="upper right",
title="Leyenda"
)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## A2.19. Exportación del dataset geoespacial enriquecido


In [ ]:
gdf_utm.to_csv(
"dataset_geoespacial_hortaleza.csv",
index=False
)
print("Dataset geoespacial guardado correctamente.")
print("Archivo: dataset_geoespacial_hortaleza.csv")
print("Dimensiones:", gdf_utm.shape)
print("\nVariables geoespaciales incorporadas:")
print(variables_osm)


## A2.20. Carga y comprobación inicial del dataset enriquecido


In [ ]:
import pandas as pd

df = pd.read_csv(
"dataset_geoespacial_hortaleza.csv"
)
print("Dimensiones del dataset:", df.shape)
print("\nNúmero de viviendas:", len(df))
display(df.head())


## A2.21. Comprobación de los tipos de datos de las variables


In [ ]:
print("Tipos de datos:")
print(df.dtypes)


## A2.22. Análisis de valores nulos del dataset


In [ ]:
nulos = df.isnull().sum()
porcentaje_nulos = (
df.isnull().mean() * 100
).round(2)
tabla_nulos = pd.DataFrame({
"Valores nulos": nulos,
"Porcentaje (%)": porcentaje_nulos
})
tabla_nulos = tabla_nulos[
tabla_nulos["Valores nulos"] > 0
].sort_values(
by="Valores nulos",
ascending=False
)
display(tabla_nulos)


## A2.23. Comprobación de registros duplicados y de identificadores propertyCode


In [ ]:
print(
"Registros completamente duplicados:",
df.duplicated().sum()
)
print(
"propertyCode duplicados:",
df["propertyCode"].duplicated().sum()
)


## A2.24. Número de valores distintos por variable


In [ ]:
n_unicos = df.nunique(
dropna=False
).sort_values()
display(n_unicos)


## A2.25. Estadísticos descriptivos de las variables numéricas


In [ ]:
display(
df.describe().T
)


## A2.26. Identificación de las variables categóricas del dataset


In [ ]:
variables_categoricas = df.select_dtypes(
include=["object", "bool"]
).columns.tolist()
print("Variables categóricas:")
print(variables_categoricas)


## A2.27. Análisis de frecuencias de las variables categóricas


In [ ]:
for variable in variables_categoricas:
print("\n" + "=" * 60)
print(variable)
print("=" * 60)
display(df[variable].value_counts(dropna=False))


## A2.28. Representación gráfica de la distribución de las principales variables numéricas


In [ ]:

import matplotlib.pyplot as plt
fig, axes = plt.subplots(
2, 2,
figsize=(12, 8)
)
axes[0, 0].hist(
df["price"].dropna() / 1000,
bins=30
)
axes[0, 0].set_title("Precio de la vivienda")
axes[0, 0].set_xlabel("Precio (miles de €)")
axes[0, 0].set_ylabel("Frecuencia")
axes[0, 0].grid(alpha=0.3)
axes[0, 1].hist(
df["size"].dropna(),
bins=30
)
axes[0, 1].set_title("Superficie de la vivienda")
axes[0, 1].set_xlabel("Superficie (m²)")
axes[0, 1].set_ylabel("Frecuencia")
axes[0, 1].grid(alpha=0.3)
frecuencia_rooms = (
df["rooms"]
.value_counts()

.sort_index()
)
axes[1, 0].bar(
frecuencia_rooms.index,
frecuencia_rooms.values
)
axes[1, 0].set_title("Número de habitaciones")
axes[1, 0].set_xlabel("Habitaciones")
axes[1, 0].set_ylabel("Frecuencia")
axes[1, 0].grid(
axis="y",
alpha=0.3
)
frecuencia_bathrooms = (
df["bathrooms"]
.value_counts()
.sort_index()
)
axes[1, 1].bar(
frecuencia_bathrooms.index,
frecuencia_bathrooms.values
)
axes[1, 1].set_title("Número de baños")
axes[1, 1].set_xlabel("Baños")
axes[1, 1].set_ylabel("Frecuencia")
axes[1, 1].grid(
axis="y",
alpha=0.3
)
plt.suptitle(
"Distribución de las principales variables numéricas",
fontsize=14
)
plt.tight_layout()
plt.show()


## A2.29. Cálculo y análisis del porcentaje de valores nulos por variable


In [ ]:
tabla_nulos = pd.DataFrame({
"Nulos": df.isnull().sum(),
"Porcentaje (%)": (df.isnull().mean() * 100).round(2)
})
tabla_nulos = tabla_nulos[
tabla_nulos["Nulos"] > 0
].sort_values(

"Porcentaje (%)",
ascending=False
)
display(tabla_nulos)


## A2.30. Análisis de frecuencias de las principales variables con valores nulos


In [ ]:

variables_nulos = [
"floor",
"exterior",
"hasLift",
"parkingSpace",
"status",
"newDevelopmentFinished",
"luxuryVisibility"
]
for variable in variables_nulos:
print("\n" + "=" * 50)
print(variable)
print("=" * 50)
display(
df[variable].value_counts(dropna=False)
)


## A2.31. Análisis de la relación entre valores nulos y tipología del inmueble


In [ ]:
# Relación entre valores nulos y tipología del inmueble
for variable in [
"floor",
"exterior",
"hasLift"
]:
tabla = pd.crosstab(
df["propertyType"],
df[variable].isna(),
normalize="index"
) * 100
print("\n" + "=" * 60)
print(f"Valores nulos en {variable} según propertyType (%)")
print("=" * 60)
display(tabla.round(2))


## A2.32. Representación mediante gráficos circulares de la proporción de valores nulos en als variables floor, exterior y hasLift 


In [ ]:
import matplotlib.pyplot as plt

variables = [
"floor",
"exterior",
"hasLift"
]
fig, axes = plt.subplots(
1,
3,
figsize=(14, 5)
)
for ax, variable in zip(axes, variables):
nulos = df[variable].isna().sum()
no_nulos = df[variable].notna().sum()
valores = [
no_nulos,
nulos
]
etiquetas = [
"Con información",
"Sin información"
]
ax.pie(
valores,
labels=etiquetas,
autopct="%1.1f%%",
startangle=90
)
ax.set_title(variable)
plt.suptitle(
"Distribución de valores nulos en variables seleccionadas",
fontsize=14
)
plt.tight_layout()
plt.show()


## A2.33. Detección de valores potencialmente atípicos mediante el rango IQR


In [ ]:

variables_numericas = [
"price",
"size",
"rooms",
"bathrooms",
"dist_metro",

"dist_bus",
"dist_school",
"dist_hospital",
"dist_health_center",
"dist_supermarket",
"dist_park",
"dist_sports"
]
resultados_outliers = []
for variable in variables_numericas:
q1 = df[variable].quantile(0.25)
q3 = df[variable].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
outliers = (
(df[variable] < limite_inferior) |
(df[variable] > limite_superior)
).sum()
porcentaje = (
outliers / len(df) * 100
)
resultados_outliers.append({
"Variable": variable,
"Q1": q1,
"Q3": q3,
"IQR": iqr,
"Límite inferior": limite_inferior,
"Límite superior": limite_superior,
"Valores atípicos": outliers,
"Porcentaje (%)": porcentaje
})
tabla_outliers = pd.DataFrame(
resultados_outliers
)
tabla_outliers["Porcentaje (%)"] = (
tabla_outliers["Porcentaje (%)"]
.round(2)
)
display(tabla_outliers)


## A2.34. Representación mediante diagramas de caja de las principales variables numéricas para la identificación visual de posibles valores atípicos


In [ ]:

import matplotlib.pyplot as plt

variables_outliers = [
"price",
"size",
"rooms",
"bathrooms"
]
fig, axes = plt.subplots(
2,
2,
figsize=(12, 8)
)
for ax, variable in zip(
axes.ravel(),
variables_outliers
):
ax.boxplot(
df[variable].dropna(),
vert=False
)
ax.set_title(variable)
ax.set_xlabel("Valor")
ax.grid(
axis="x",
alpha=0.3
)
plt.suptitle(
"Detección visual de posibles valores atípicos",
fontsize=14
)
plt.tight_layout()
plt.show()


## A2.35. Inspección de las observaciones identificadas como potencialmente atípicas en las principales variables inmobiliarias


In [ ]:

variables_revision = [
    "price",
    "size",
    "rooms",
    "bathrooms"
]

for variable in variables_revision:
    q1 = df[variable].quantile(0.25)
    q3 = df[variable].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    extremos = df[
        (df[variable] < limite_inferior) |
        (df[variable] > limite_superior)
    ].copy()

    print("\n" + "=" * 70)
    print(f"Valores extremos detectados en: {variable}")
    print("=" * 70)

    columnas_mostrar = [
        "propertyCode",
        "propertyType",
        "price",
        "size",
        "rooms",
        "bathrooms"
    ]

    display(
        extremos[
            columnas_mostrar
        ].sort_values(
            by=variable,
            ascending=False
        ).head(15)
    )


## A2.36. Análisis de la distribución de los valores potencialmente atípicos según la tipología del inmueble


In [ ]:

variables_revision = [
"price",
"size",
"rooms",
"bathrooms"
]
for variable in variables_revision:

q1 = df[variable].quantile(0.25)
q3 = df[variable].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
extremos = df[
(df[variable] < limite_inferior) |
(df[variable] > limite_superior)
].copy()
print("\n" + "=" * 70)
print(f"Valores extremos detectados en: {variable}")
print("=" * 70)
columnas_mostrar = [
"propertyCode",
"propertyType",
"price",
"size",
"rooms",
"bathrooms"
]
display(
extremos[
columnas_mostrar
].sort_values(
by=variable,
ascending=False
).head(15)
)


## A2.37. Revisión de los valores y tipos de datos de la variable floor


In [ ]:
print("Tipo de dato:")
print(df["floor"].dtype)

print("\nValores registrados en floor:")
display(
    df["floor"]
    .value_counts(dropna=False)
    .sort_index()
)


## A2.38. Revisión de las categorías y estructura de la variable detailedType


In [ ]:
print("Valores distintos de detailedType:")
display(

    df["detailedType"]
    .value_counts(dropna=False)
)


## A2.39. Identificación de variables sin variabilidad dentro del dataset


In [ ]:
# Identificación de variables sin variabilidad
n_unicos = (
    df.nunique(dropna=False)
    .sort_values()
)

variables_constantes = n_unicos[
    n_unicos == 1
]

print("Variables con un único valor:")
display(
    variables_constantes
)


## A2.40. Carga del dataset geoespacial


In [ ]:
import pandas as pd
df = pd.read_csv(
"dataset_geoespacial_hortaleza.csv"
)
print("Dimensiones del dataset:", df.shape)
print("\nNúmero de viviendas:", len(df))
display(df.head())


## A2.41. Creación de una copia de trabajo


In [ ]:
df_model = df.copy()
print("Dimensiones del dataset de trabajo:", df_model.shape)
print("\nNúmero de viviendas:", len(df_model))


## A2.42. Extracción de la variable typology


In [ ]:
import ast
def extraer_typology(valor):
if pd.isna(valor):
return pd.NA
try:
datos = ast.literal_eval(valor)
return datos.get("typology", pd.NA)
except (ValueError, SyntaxError):

return pd.NA
df_model["typology"] = df_model["detailedType"].apply(
extraer_typology
)
print("Valores de typology:")
display(
df_model["typology"].value_counts(dropna=False)
)


## A2.43. Definición de las variables seleccionadas


In [ ]:
variable_objetivo = ["price"]
variables_estructurales = [
"size", "rooms", "bathrooms", "floor", "hasLift",
"status", "propertyType", "typology", "exterior",
"newDevelopment"
]
variables_geograficas = ["latitude", "longitude"]
variables_geoespaciales = [
"dist_metro", "dist_bus", "dist_school", "dist_hospital",
"dist_health_center", "dist_supermarket", "dist_park",
"dist_sports"
]
variables_modelo = (
variable_objetivo
+ variables_estructurales
+ variables_geograficas
+ variables_geoespaciales
)
print("Variable objetivo:")
print(variable_objetivo)
print("\nVariables estructurales:")
print(variables_estructurales)
print("\nVariables geográficas:")
print(variables_geograficas)
print("\nVariables geoespaciales:")
print(variables_geoespaciales)
print("\nNúmero total de variables seleccionadas:")
print(len(variables_modelo))


## A2.44. Revisión de variables categóricas con valores nulos


In [ ]:
variables_categoricas_nulos = [
"floor", "hasLift", "status", "exterior"
]
for variable in variables_categoricas_nulos:

print("\n" + "=" * 60)
print(variable)
print("=" * 60)
display(
df_modelo[variable].value_counts(dropna=False)
)


## A2.45. Tratamiento de valores ausentes en variables categóricas


In [ ]:
variables_categoricas_nulos = [
"floor", "hasLift", "status", "exterior"
]
for variable in variables_categoricas_nulos:
df_modelo[variable] = (
df_modelo[variable]
.astype("object")
.where(df_modelo[variable].notna(), "unknown")
)
print("Comprobación de valores nulos:")
display(
df_modelo[variables_categoricas_nulos].isnull().sum()
)
print("\nFrecuencias después del tratamiento:")
for variable in variables_categoricas_nulos:
print("\n" + "=" * 60)
print(variable)
print("=" * 60)
display(
df_modelo[variable].value_counts(dropna=False)
)


## A2.46. Revisión de las variables categóricas seleccionadas


In [ ]:
variables_categoricas = [
"floor", "hasLift", "status", "propertyType",
"typology", "exterior", "newDevelopment"
]
for variable in variables_categoricas:
print("\n" + "=" * 60)
print(variable)
print("=" * 60)
print("Tipo de dato:", df_modelo[variable].dtype)
display(
df_modelo[variable].value_counts(dropna=False)
)


## A2.47. Conversión de newDevelopment a variable binaria


In [ ]:
df_modelo["newDevelopment"] = (
df_modelo["newDevelopment"].astype(int)
)
print("Frecuencias de newDevelopment:")
display(
df_modelo["newDevelopment"].value_counts().sort_index()
)
print("\nTipo de dato:")
print(df_modelo["newDevelopment"].dtype)


## A2.48. Normalización de la variable floor


In [ ]:
df_modelo["floor"] = (
df_modelo["floor"].astype(str).str.strip()
)
print("Valores de floor después de la normalización:")
display(
df_modelo["floor"].value_counts()
)


## A2.49. Preparación de las variables categóricas


In [ ]:
variables_categoricas = [
"hasLift", "status", "propertyType",
"typology", "exterior"
]
for variable in variables_categoricas:
df_modelo[variable] = (
df_modelo[variable].astype(str).str.strip()
)
print("Tipos de datos después de la preparación:")
display(
df_modelo[variables_categoricas + ["floor"]].dtypes
)


## A2.50. Clasificación de variables según su tipo


In [ ]:
variables_numericas = [
"price", "size", "rooms", "bathrooms", "newDevelopment",
"latitude", "longitude", "dist_metro", "dist_bus",
"dist_school", "dist_hospital", "dist_health_center",
"dist_supermarket", "dist_park", "dist_sports"
]
variables_categoricas = [
"floor", "hasLift", "status",

"propertyType", "typology", "exterior"
]
print("Variables numéricas:")
print(variables_numericas)
print("\nNúmero de variables numéricas:", len(variables_numericas))
print("\nVariables categóricas:")
print(variables_categoricas)
print("\nNúmero de variables categóricas:", len(variables_categoricas))
print("\nComprobación del número total de variables:")
print(
"Total:",
len(variables_numericas) + len(variables_categoricas)
)


## A2.51. Separación de la variable objetivo y las variables predictoras


In [ ]:
variable_objetivo = "price"
variables_predictoras = [
variable
for variable in variables_modelo
if variable != variable_objetivo
]
X = df_modelo[variables_predictoras].copy()
y = df_modelo[variable_objetivo].copy()
print("Variable objetivo:")
print(variable_objetivo)
print("\nNúmero de variables predictoras:")
print(len(variables_predictoras))
print("\nVariables predictoras:")
print(variables_predictoras)
print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)


## A2.52. Clasificación de las variables predictoras


In [ ]:
variables_numericas_predictoras = [
"size", "rooms", "bathrooms", "newDevelopment",
"latitude", "longitude", "dist_metro", "dist_bus",
"dist_school", "dist_hospital", "dist_health_center",
"dist_supermarket", "dist_park", "dist_sports"
]
variables_categoricas_predictoras = [
"floor", "hasLift", "status",
"propertyType", "typology", "exterior"
]
print("Variables numéricas predictoras:")
print(variables_numericas_predictoras)

print("\nNúmero de variables numéricas:", len(variables_numericas_predictoras))
print("\nVariables categóricas predictoras:")
print(variables_categoricas_predictoras)
print("\nNúmero de variables categóricas:", len(variables_categoricas_predictoras))
print("\nTotal de variables predictoras:",
len(variables_numericas_predictoras) + len(variables_categoricas_predictoras))


## A2.53. Control de calidad previo a la codificación


In [ ]:
print("Valores nulos en variables numéricas:")
display(
X[variables_numericas_predictoras].isnull().sum()
)
print("\nValores nulos en variables categóricas:")
display(
X[variables_categoricas_predictoras].isnull().sum()
)
print("\nTipos de datos de las variables predictoras:")
display(
X[
variables_numericas_predictoras
+ variables_categoricas_predictoras
].dtypes
)


## A2.54. Configuración del preprocesamiento de variables


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocesador = ColumnTransformer(
transformers=[
(
"categoricas",
OneHotEncoder(
handle_unknown="ignore",
sparse_output=False
),
variables_categoricas_predictoras
)
],
remainder="passthrough"
)
print("Preprocesador configurado correctamente.")
print("\nVariables categóricas a codificar:")
print(variables_categoricas_predictoras)
print("\nVariables numéricas que se mantienen:")

print(variables_numericas_predictoras)


## A2.55. Aplicación y comprobación del preprocesamiento


In [ ]:
X_transformado = preprocesador.fit_transform(X)
print("Dimensiones de X antes del preprocesamiento:")
print(X.shape)
print("\nDimensiones de X después del preprocesamiento:")
print(X_transformado.shape)
print("\nTipo de objeto resultante:")
print(type(X_transformado))


## A2.56. Comprobación de las variables generadas


In [ ]:
nombres_variables_transformadas = (
preprocesador.get_feature_names_out()
)
print("Número de variables generadas:",
len(nombres_variables_transformadas))
print("\nVariables generadas:")
for i, variable in enumerate(
nombres_variables_transformadas, start=1
):
print(f"{i}. {variable}")


## A2.57. Control final del dataset preparado


In [ ]:
print("Dimensiones finales del dataset de modelización:")
print(df_modelo.shape)
print("\nRegistros completamente duplicados:")
print(df_modelo.duplicated().sum())
print("\nValores nulos en la variable objetivo:")
print(df_modelo["price"].isnull().sum())
print("\nValores nulos en las variables predictoras:")
print(
df_modelo[variables_predictoras].isnull().sum().sum()
)
print("\nNúmero de variables predictoras:")
print(len(variables_predictoras))
print("\nNúmero de viviendas:")
print(len(df_modelo))


## A2.58. Identificación de registros coincidentes


In [ ]:
duplicados_modelo = df_modelo[
df_modelo.duplicated(keep=False)
].copy()
print("Número de registros implicados:",

len(duplicados_modelo))
display(
df.loc[
duplicados_modelo.index,
[
"propertyCode",
"externalReference",
"price",
"size",
"rooms",
"bathrooms",
"propertyType",
"latitude",
"longitude"
]
]
)


## A2.59. Control final de identificadores


In [ ]:
print("Número de viviendas:", len(df_modelo))
print("\nNúmero de propertyCode distintos:")
print(df.loc[df_modelo.index, "propertyCode"].nunique())
print("\npropertyCode duplicados:")
print(
df.loc[df_modelo.index, "propertyCode"].duplicated().sum()
)
print("\nRegistros completos idénticos en las variables de modelización:")
print(df_modelo.duplicated().sum())


## A2.60. Exportación del dataset preparado


In [ ]:
nombre_archivo = "dataset_modelo_hortaleza.csv"
df_modelo.to_csv(
nombre_archivo,
index=False
)
print("Dataset preparado exportado correctamente.")
print("Archivo:", nombre_archivo)
print("Dimensiones:", df_modelo.shape)


## A2.61. Comprobación final del dataset exportado


In [ ]:
df_final = pd.read_csv(
"dataset_modelo_hortaleza.csv"
)
print("Dimensiones del dataset exportado:")

print(df_final.shape)
print("\nNúmero de viviendas:")
print(len(df_final))
print("\nVariables:")
print(df_final.columns.tolist())
print("\nValores nulos:")
print(df_final.isnull().sum().sum())
